# 01 — Data Cleaning

**Purpose:** Prepare raw build-system metrics for downstream analysis.  
Steps include missing-data imputation, outlier detection, path alignment,
target validation against the dependency graph, type casting, and
normalisation. Cleaned datasets are saved back to Parquet.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from build_optimiser.graph import load_graph, attach_metrics
from build_optimiser.config import load_config

cfg = load_config(PROJECT_ROOT / "config.yaml")

# Load processed data
file_metrics = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "file_metrics.parquet")
target_metrics = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "target_metrics.parquet")
edge_list = pd.read_parquet(PROJECT_ROOT / "data" / "processed" / "edge_list.parquet")

# Load dependency graph
G = load_graph(str(PROJECT_ROOT / "data" / "raw" / "dot"))
attach_metrics(G, target_metrics)

print(f"Files: {len(file_metrics)}, Targets: {len(target_metrics)}, Edges: {len(edge_list)}")

In [ ]:
# ── Missing data analysis ─────────────────────────────────────────────
print("=== Missing values per column (target_metrics) ===")
tm_missing = target_metrics.isnull().sum()
print(tm_missing[tm_missing > 0].sort_values(ascending=False))
print()

print("=== Missing values per column (file_metrics) ===")
fm_missing = file_metrics.isnull().sum()
print(fm_missing[fm_missing > 0].sort_values(ascending=False))
print()

# Targets with missing compile times
if "compile_time_s" in target_metrics.columns:
    missing_ct = target_metrics[target_metrics["compile_time_s"].isna()]
    print(f"Targets missing compile_time_s: {len(missing_ct)} / {len(target_metrics)}")
    if len(missing_ct) > 0:
        print(missing_ct.head(10))
else:
    print("Column 'compile_time_s' not found — check schema.")

# Visualise missingness pattern
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(target_metrics.isnull().T, cbar=False, cmap="viridis", ax=ax)
ax.set_title("Missing-value pattern (target_metrics)")
ax.set_xlabel("Row index")
plt.tight_layout()
plt.show()

In [ ]:
# ── Outlier detection (IQR method on compile times) ───────────────────
def flag_iqr_outliers(series: pd.Series, k: float = 1.5) -> pd.Series:
    """Return boolean mask where True = outlier."""
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return (series < lower) | (series > upper)

if "compile_time_s" in target_metrics.columns:
    ct = target_metrics["compile_time_s"].dropna()
    outlier_mask = flag_iqr_outliers(ct)
    n_outliers = outlier_mask.sum()
    print(f"IQR outliers (k=1.5): {n_outliers} / {len(ct)} ({100*n_outliers/len(ct):.1f}%)")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].boxplot(ct, vert=False)
    axes[0].set_xlabel("compile_time_s")
    axes[0].set_title("Box plot (before cleaning)")

    axes[1].hist(ct[~outlier_mask], bins=50, edgecolor="black", alpha=0.7, label="inliers")
    axes[1].hist(ct[outlier_mask], bins=50, edgecolor="black", alpha=0.7, color="red", label="outliers")
    axes[1].set_xlabel("compile_time_s")
    axes[1].set_ylabel("count")
    axes[1].set_title("Histogram with outliers highlighted")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

    # Tag outliers — keep rows but add a flag column
    target_metrics["compile_time_outlier"] = False
    target_metrics.loc[ct.index[outlier_mask], "compile_time_outlier"] = True
else:
    print("Skipping outlier detection — compile_time_s not available.")

In [ ]:
# ── Path alignment verification ──────────────────────────────────────
# Ensure all file paths in file_metrics use POSIX separators and are
# relative to the project root stored in the config.

path_col = "path" if "path" in file_metrics.columns else file_metrics.columns[0]
print(f"Using column '{path_col}' as file-path identifier.")

# Normalise to forward slashes
file_metrics[path_col] = file_metrics[path_col].astype(str).str.replace("\\\\", "/", regex=False)

# Strip any leading './' or absolute prefix matching project root
source_root = cfg.get("source_root", "")
if source_root:
    file_metrics[path_col] = file_metrics[path_col].str.replace(
        f"^{source_root}/?", "", regex=True
    )
file_metrics[path_col] = file_metrics[path_col].str.lstrip("./")

print(f"Sample paths after alignment:")
print(file_metrics[path_col].head(10).to_string())

In [ ]:
# ── Target validation (cross-reference DOT files with build tree) ────
graph_targets = set(G.nodes())
name_col = "target" if "target" in target_metrics.columns else target_metrics.columns[0]
metric_targets = set(target_metrics[name_col].astype(str))

in_graph_not_metrics = graph_targets - metric_targets
in_metrics_not_graph = metric_targets - graph_targets

print(f"Targets in graph only      : {len(in_graph_not_metrics)}")
print(f"Targets in metrics only    : {len(in_metrics_not_graph)}")
print(f"Targets in both            : {len(graph_targets & metric_targets)}")

if in_graph_not_metrics:
    print("\nSample graph-only targets:")
    for t in sorted(in_graph_not_metrics)[:10]:
        print(f"  {t}")

if in_metrics_not_graph:
    print("\nSample metrics-only targets (will be dropped):")
    for t in sorted(in_metrics_not_graph)[:10]:
        print(f"  {t}")

# Drop rows from target_metrics that are not present in the graph
before = len(target_metrics)
target_metrics = target_metrics[target_metrics[name_col].astype(str).isin(graph_targets)].copy()
print(f"\nDropped {before - len(target_metrics)} orphan targets from target_metrics.")

In [ ]:
# ── Type casting and normalisation ───────────────────────────────────
# Ensure numeric columns are proper dtypes and normalise key metrics.

numeric_candidates = [
    "compile_time_s", "link_time_s", "sloc", "header_count",
    "header_depth", "object_size_bytes",
]

for col in numeric_candidates:
    if col in target_metrics.columns:
        target_metrics[col] = pd.to_numeric(target_metrics[col], errors="coerce")

for col in numeric_candidates:
    if col in file_metrics.columns:
        file_metrics[col] = pd.to_numeric(file_metrics[col], errors="coerce")

# Min-max normalisation for selected columns (store as new _norm cols)
norm_cols = [c for c in ["compile_time_s", "sloc", "header_depth"] if c in target_metrics.columns]
for col in norm_cols:
    series = target_metrics[col]
    lo, hi = series.min(), series.max()
    if hi - lo > 0:
        target_metrics[f"{col}_norm"] = (series - lo) / (hi - lo)
    else:
        target_metrics[f"{col}_norm"] = 0.0

print("Dtypes after casting:")
print(target_metrics.dtypes)
print("\nDescriptive statistics (normalised cols):")
norm_col_names = [c for c in target_metrics.columns if c.endswith("_norm")]
if norm_col_names:
    print(target_metrics[norm_col_names].describe())

In [ ]:
# ── Save cleaned datasets ────────────────────────────────────────────
out_dir = PROJECT_ROOT / "data" / "cleaned"
out_dir.mkdir(parents=True, exist_ok=True)

target_metrics.to_parquet(out_dir / "target_metrics.parquet", index=False)
file_metrics.to_parquet(out_dir / "file_metrics.parquet", index=False)
edge_list.to_parquet(out_dir / "edge_list.parquet", index=False)

print(f"Saved cleaned datasets to {out_dir}")
print(f"  target_metrics : {len(target_metrics)} rows")
print(f"  file_metrics   : {len(file_metrics)} rows")
print(f"  edge_list      : {len(edge_list)} rows")

## Codegen Data Cleaning

Validate the code-generation inventory and ensure generated files are correctly
mapped to targets and file metrics.

In [ ]:
# ── Review unknown generators ────────────────────────────────────────
# Print all codegen entries classified as 'unknown_codegen' for manual
# review.  Decide whether to reclassify to a known generator, create a
# new named generator, or reclassify as non_codegen.

codegen_path = PROJECT_ROOT / "data" / "raw" / "codegen_inventory.csv"
if codegen_path.exists():
    codegen_inv = pd.read_csv(codegen_path)
    unknown = codegen_inv[codegen_inv["generator"] == "unknown_codegen"]
    print(f"Unknown codegen entries: {len(unknown)} / {len(codegen_inv)}")
    if not unknown.empty:
        print("\nReview these commands and reclassify as needed:")
        for _, row in unknown.iterrows():
            print(f"  Command : {row['command'][:120]}")
            print(f"  Outputs : {row['output_files'][:120]}")
            print(f"  Desc    : {row['description']}")
            print()
    else:
        print("No unknown generators found — all custom commands classified.")
else:
    codegen_inv = pd.DataFrame()
    print("codegen_inventory.csv not found — skipping codegen cleaning.")

In [ ]:
# ── Validate generator-to-file mapping ───────────────────────────────
# 1. Generated files in codegen inventory that don't appear in file_metrics
#    (generated but never compiled — possibly dead code).
# 2. Files that look generated by naming pattern but are NOT in the inventory
#    (pre-generated files committed to repo, or missed classifier patterns).

if not codegen_inv.empty and "is_generated" in file_metrics.columns:
    # Explode codegen inventory outputs
    inv_outputs = set()
    for _, row in codegen_inv.iterrows():
        if pd.notna(row["output_files"]):
            inv_outputs.update(o.strip() for o in str(row["output_files"]).split(";") if o.strip())

    metric_files = set(file_metrics["source_file"].dropna())

    orphan_generated = inv_outputs - metric_files
    print(f"Generated files NOT in file_metrics (potential dead code): {len(orphan_generated)}")
    for f in sorted(orphan_generated)[:15]:
        print(f"  {f}")
    if len(orphan_generated) > 15:
        print(f"  ... and {len(orphan_generated) - 15} more")

    # Files that look generated but aren't in the inventory
    gen_patterns = (".pb.cc", ".pb.h", ".grpc.pb.cc", ".grpc.pb.h")
    suspect = [
        f for f in metric_files
        if any(f.endswith(p) for p in gen_patterns) or "/generated/" in f
    ]
    known_generated = set(
        file_metrics.loc[file_metrics["is_generated"] == True, "source_file"]
    )
    missed = [f for f in suspect if f not in known_generated]
    print(f"\nFiles with generated-looking names NOT in codegen inventory: {len(missed)}")
    for f in sorted(missed)[:15]:
        print(f"  {f}")
else:
    print("Skipping — codegen inventory or is_generated column not available.")

In [ ]:
# ── Validate target ownership ────────────────────────────────────────
# Check for codegen entries where cmake_target is empty (generated files
# that couldn't be mapped to a target).

if not codegen_inv.empty:
    empty_target = codegen_inv[
        codegen_inv["cmake_target"].isna() | (codegen_inv["cmake_target"].astype(str).str.strip() == "")
    ]
    print(f"Codegen entries with no cmake_target: {len(empty_target)} / {len(codegen_inv)}")
    if not empty_target.empty:
        print("\nThese generated files could not be mapped to a target:")
        for _, row in empty_target.iterrows():
            print(f"  Generator: {row['generator']}")
            print(f"  Outputs  : {row['output_files'][:100]}")
            print()
else:
    print("Skipping — codegen inventory not available.")

In [ ]:
# ── Handle generator timing nulls ────────────────────────────────────
# If gen_time_ms is null for codegen entries (no build had completed when
# the inventory was collected), and a .ninja_log exists, backfill timings.

if not codegen_inv.empty:
    null_timing = codegen_inv[codegen_inv["gen_time_ms"].isna() | (codegen_inv["gen_time_ms"] == "")]
    print(f"Codegen entries with null gen_time_ms: {len(null_timing)} / {len(codegen_inv)}")

    ninja_log_path = Path(cfg["build_dir"]) / ".ninja_log"
    if len(null_timing) > 0 and ninja_log_path.exists():
        print(f"Ninja log found at {ninja_log_path} — attempting backfill...")
        from build_optimiser.codegen import parse_ninja_log_for_commands

        all_outputs = set()
        for _, row in null_timing.iterrows():
            if pd.notna(row["output_files"]):
                all_outputs.update(o.strip() for o in str(row["output_files"]).split(";") if o.strip())

        timings = parse_ninja_log_for_commands(str(ninja_log_path), all_outputs)
        backfilled = 0
        for idx, row in codegen_inv.iterrows():
            if pd.isna(row["gen_time_ms"]) or row["gen_time_ms"] == "":
                outputs = str(row["output_files"]).split(";") if pd.notna(row["output_files"]) else []
                for out in outputs:
                    out = out.strip()
                    if out in timings:
                        codegen_inv.at[idx, "gen_time_ms"] = timings[out]
                        backfilled += 1
                        break
        print(f"Backfilled {backfilled} timing entries from ninja log.")

        if backfilled > 0:
            codegen_inv.to_csv(codegen_path, index=False)
            print("Updated codegen_inventory.csv with backfilled timings.")
    elif len(null_timing) > 0:
        print("No ninja log available — timings will remain null until a build completes.")
else:
    print("Skipping — codegen inventory not available.")